# Lab 02 · Publish a Foundry model and make your first call

Publish a Microsoft Foundry model deployment through your AI Gateway, create a runtime API key, and call the model
with the OpenAI Python SDK. This notebook is the Python track of the [lab guide](README.md): it creates the same
provider, backend model, alias, and key as the CLI track, then runs the same checks.

**Create Foundry deployment → Grant gateway identity → Publish model → Create key → Call → Verify**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Foundry account and `gpt-4.1-mini` deployment | The backend that serves the model. The notebook creates one if it doesn't exist. |
| Foundry User role | Lets the gateway's managed identity call the account, so no Foundry keys are involved. |
| Model provider `foundry-models` | Tells the gateway where the account is and to authenticate with its managed identity. |
| Backend model `gpt-4.1-mini` | Registers the deployment under the provider. |
| Alias `gpt-4.1-mini` | The stable name clients send in the `model` field. |
| Runtime API key `app-first-call` | Authenticates your app to the gateway. Notebooks 04 and 06 reuse it. |

![Request lifecycle: the gateway validates the key, runs policies, swaps in its managed identity, and forwards to Foundry](../assets/diagrams/request-lifecycle.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  reuses your CLI sign-in through `AzureCliCredential`.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) has run, so the gateway exists.
- Quota for `gpt-4.1-mini` (`GlobalStandard`) in the gateway's region. Model inference is billed per use.

Run the cells in order, or click **Run All**. Each step is idempotent: it reuses what already exists.

In [ ]:
%pip install -q --upgrade azure-identity requests openai

### 0. Initialize variables and clients

The defaults match notebook 01, so the cell finds the gateway without edits, and stops with a clear message if it
isn't there. It also reads the gateway's managed identity, which needs access to Foundry. If you already have a Foundry
account with a `gpt-4.1-mini` deployment, set `foundry_account` and `foundry_rg` to use it instead of creating one.

In [ ]:
import hashlib, json, os, re, subprocess, sys, time, uuid
from pathlib import Path

import requests
from azure.identity import AzureCliCredential
from openai import APIStatusError, OpenAI

location        = os.environ.get("AIGW_LOCATION", "eastus2")                  # change to swedencentral if you prefer
resource_group  = os.environ.get("AIGW_RESOURCE_GROUP", "rg-ai-gateway-labs") # change to use another resource group
gateway_name    = os.environ.get("AIGW_GATEWAY_NAME", "")                     # change to match notebook 01 (empty = derived)
foundry_account = os.environ.get("AIGW_FOUNDRY_ACCOUNT", "")                  # change to reuse a Foundry account (empty = derived)
foundry_rg      = os.environ.get("AIGW_FOUNDRY_RESOURCE_GROUP", resource_group)
deployment      = "gpt-4.1-mini"     # Foundry deployment, backend model, and alias name
model_version   = "2025-04-14"
provider        = "foundry-models"   # gateway model provider
api_key_name    = "app-first-call"   # runtime API key, reused by notebooks 04 and 06

lab_name   = Path(globals().get("__vsc_ipynb_file__", "02-first-model-call.ipynb")).stem
lab_labels = {"lab": lab_name, "workload": "ai-gateway-labs"}   # tags for clean-up

result = subprocess.run("az account show -o json", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    sys.exit("❌ Azure CLI not installed or not signed in. Run `az login` first.")
account         = json.loads(result.stdout)
subscription_id = account["id"]
tenant_id       = account["tenantId"]
user_name       = account["user"]["name"]

suffix       = hashlib.sha1(subscription_id.encode()).hexdigest()   # stable per subscription
gateway_name = gateway_name or f"aigw-labs-{suffix[:6]}"
assert location in ("eastus2", "swedencentral"), "During the preview, AI Gateway runs in eastus2 or swedencentral."
assert re.fullmatch(r"[A-Za-z]([A-Za-z0-9-]{0,22}[A-Za-z0-9])?", gateway_name), "Use 1-24 letters, digits, or hyphens."

ARM        = "https://management.azure.com"
ARM_API    = "2025-09-01-preview"   # Microsoft.ApiManagement/aigateways
GATEWAY_ID = (f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
              f"/providers/Microsoft.ApiManagement/aigateways/{gateway_name}")

credential = AzureCliCredential()   # every client below reuses your az login
_arm_token = None

def arm(method, path, body=None, api_version=ARM_API, params=None):
    """Call Azure Resource Manager and return the JSON body (None when a GET or DELETE finds nothing)."""
    global _arm_token
    if _arm_token is None or _arm_token.expires_on - time.time() < 300:
        _arm_token = credential.get_token("https://management.azure.com/.default")
    response = requests.request(method, ARM + path, json=body, timeout=120,
                                params={"api-version": api_version, **(params or {})},
                                headers={"Authorization": f"Bearer {_arm_token.token}"})
    if response.status_code == 404 and method in ("GET", "DELETE"):
        return None
    if not response.ok:
        raise RuntimeError(f"{method} {path} -> {response.status_code}: {response.text[:1000]}")
    return response.json() if response.content else {}

def wait_until_ready(path, api_version=ARM_API, label="Resource", timeout_minutes=30):
    """Poll properties.provisioningState until it reports Succeeded."""
    start = time.time()
    while True:
        resource = arm("GET", path, api_version=api_version)
        state = "NotFound" if resource is None else resource.get("properties", {}).get("provisioningState", "Succeeded")
        if state == "Succeeded":
            return resource
        if state in ("Failed", "Canceled"):
            raise RuntimeError(f"{label} provisioning {state}: {json.dumps(resource.get('properties'))[:1000]}")
        if time.time() - start > timeout_minutes * 60:
            raise TimeoutError(f"{label} is still {state} after {timeout_minutes} minutes.")
        print(f"⏳ {label}: {state} ({int(time.time() - start)} s)")
        time.sleep(20)

gateway = arm("GET", GATEWAY_ID)
if not gateway or gateway["properties"].get("provisioningState") != "Succeeded":
    sys.exit(f"❌ Gateway {gateway_name} not found in {resource_group}. Run 01-provision-gateway.ipynb first.")
gateway_url = "https://" + gateway["properties"]["frontend"]["defaultHostname"]
WS          = f"{GATEWAY_ID}/workspaces/default"   # models, aliases, tool servers, and policies live here

principal_id = gateway["identity"]["principalId"]   # the gateway's managed identity

foundry_account = foundry_account or f"aigwlabs{suffix[:8]}"
FOUNDRY_ID      = (f"/subscriptions/{subscription_id}/resourceGroups/{foundry_rg}"
                   f"/providers/Microsoft.CognitiveServices/accounts/{foundry_account}")
COGNITIVE_API   = "2025-04-01-preview"
alias           = deployment

def assign_role(scope, role_id, principal_id, principal_type):
    """Create a role assignment with a deterministic name, so reruns reuse it."""
    name = uuid.uuid5(uuid.NAMESPACE_URL, f"{scope}|{role_id}|{principal_id}".lower())
    path = f"{scope}/providers/Microsoft.Authorization/roleAssignments/{name}"
    if arm("GET", path, api_version="2022-04-01"):
        return path, False
    body = {"properties": {
        "roleDefinitionId": f"/subscriptions/{subscription_id}/providers/Microsoft.Authorization/roleDefinitions/{role_id}",
        "principalId": principal_id, "principalType": principal_type}}
    for attempt in range(10):
        try:
            arm("PUT", path, body, api_version="2022-04-01")
            return path, True
        except RuntimeError as error:
            if "RoleAssignmentExists" in str(error):
                return path, False
            if "PrincipalNotFound" not in str(error) or attempt == 9:
                raise
            print("⏳ Waiting for the identity to replicate...")
            time.sleep(15)

print(f"🧪 Lab:            {lab_name}")
print(f"👤 User:           {user_name}")
print(f"🏛️ Tenant:         {tenant_id}")
print(f"🔑 Subscription:   {account['name']} ({subscription_id})")
print(f"🌍 Region:         {location}")
print(f"📁 Resource group: {resource_group}")
print(f"🚪 Gateway:        {gateway_name}")
print(f"🌐 Runtime URL:    {gateway_url}")
print(f"🏭 Foundry:        {foundry_account} ({foundry_rg})")
print(f"🤖 Model:          {deployment} {model_version}")

### 1. Create the Foundry account and model deployment

The gateway publishes models that already run in Foundry. This cell creates an Azure AI Services account and a
`gpt-4.1-mini` `GlobalStandard` deployment, mirroring [infra/foundry.bicep](infra/foundry.bicep). Local key
authentication stays disabled: the gateway authenticates with its managed identity. If the account and deployment
already exist, the cell reuses them.

In [ ]:
account_body = {
    "location": location,
    "kind": "AIServices",
    "sku": {"name": "S0"},
    "tags": lab_labels,
    "identity": {"type": "SystemAssigned"},
    "properties": {
        "allowProjectManagement": False,
        "customSubDomainName": foundry_account.lower(),
        "disableLocalAuth": True,
        "publicNetworkAccess": "Enabled",
        "networkAcls": {"defaultAction": "Allow", "virtualNetworkRules": [], "ipRules": []},
    },
}
if arm("GET", FOUNDRY_ID, api_version=COGNITIVE_API):
    print(f"♻️ Foundry account {foundry_account} exists")
else:
    print(f"📦 Creating Foundry account {foundry_account}")
    arm("PUT", FOUNDRY_ID, account_body, api_version=COGNITIVE_API)
foundry = wait_until_ready(FOUNDRY_ID, COGNITIVE_API, "Foundry account")
foundry_endpoint = foundry["properties"]["endpoint"].rstrip("/") + "/"   # the provider expects a trailing slash

DEPLOYMENT_ID = f"{FOUNDRY_ID}/deployments/{deployment}"
if arm("GET", DEPLOYMENT_ID, api_version=COGNITIVE_API):
    print(f"♻️ Deployment {deployment} exists")
else:
    print(f"📦 Deploying {deployment} {model_version}")
    arm("PUT", DEPLOYMENT_ID, {
        "sku": {"name": "GlobalStandard", "capacity": 10},
        "properties": {"model": {"format": "OpenAI", "name": deployment, "version": model_version}},
    }, api_version=COGNITIVE_API)
wait_until_ready(DEPLOYMENT_ID, COGNITIVE_API, "Deployment")
print(f"✅ Foundry endpoint: {foundry_endpoint}")

### 2. Grant the gateway access to Foundry

The gateway calls Foundry with its system-assigned managed identity, which needs the **Foundry User** role on the
account. The assignment name is derived from the scope, role, and principal, so a rerun finds the same assignment.
Role changes can take a few minutes to apply; the first call in step 6 retries until they do.

In [ ]:
FOUNDRY_USER = "53ca6127-db72-4b80-b1b0-d745d6d5456d"
role_assignment, created = assign_role(FOUNDRY_ID, FOUNDRY_USER, principal_id, "ServicePrincipal")
print(("✅ Assigned" if created else "♻️ Reusing") + f" Foundry User for the gateway identity on {foundry_account}")

### 3. Create the model provider and backend model

A **model provider** tells the gateway where the account is and how to authenticate to it. A **backend model**
registers one deployment under that provider; name it after the deployment. The cell writes them only if they're
missing or point somewhere else.

In [ ]:
PROVIDER_PATH = f"{WS}/modelProviders/{provider}"
MODEL_PATH    = f"{PROVIDER_PATH}/models/{deployment}"

existing = arm("GET", PROVIDER_PATH)
if existing and existing["properties"].get("foundry", {}).get("endpoint") == foundry_endpoint:
    print(f"♻️ Provider {provider} exists")
else:
    arm("PUT", PROVIDER_PATH, {"properties": {
        "kind": "Foundry",
        "displayName": "Foundry models",
        "foundry": {
            "endpoint": foundry_endpoint,
            "resourceIds": [FOUNDRY_ID],
            "authentication": {"kind": "ManagedIdentity",
                               "managedIdentity": {"resource": "https://cognitiveservices.azure.com/"}},
        },
    }})
    print(f"✅ Provider {provider} -> {foundry_endpoint}")

if arm("GET", MODEL_PATH):
    print(f"♻️ Backend model {deployment} exists")
else:
    arm("PUT", MODEL_PATH, {"properties": {
        "displayName": deployment,
        "apiFormat": "OpenAIChatCompletions",
        "supportedEndpoints": ["/openai/v1/chat/completions", "/openai/v1/responses"],
        "deployment": {"resourceId": DEPLOYMENT_ID, "modelName": deployment},
    }})
    print(f"✅ Backend model {deployment}")

### 4. Publish the alias

The **alias** is the name clients send in the request `model` field. It points at one or more backend models, so you
can retarget it later without changing client code (lab 03). Aliases are looked up by `properties.aliasName`.

In [ ]:
MODEL_ID = f"{GATEWAY_ID}/workspaces/default/modelProviders/{provider}/models/{deployment}"

aliases = arm("GET", f"{WS}/aliases")["value"]
current = next((a for a in aliases if a["properties"].get("aliasName") == alias), None)
if current is None:
    current = arm("PUT", f"{WS}/aliases/{alias}", {"properties": {"backendModels": [{"resourceId": MODEL_ID}]}})
    print(f"✅ Alias {alias} -> {deployment}")
else:
    targets = [b["resourceId"] for b in current["properties"].get("backendModels", [])]
    if any(t.lower() == MODEL_ID.lower() for t in targets):
        print(f"♻️ Alias {alias} exists")
    else:
        print(f"⚠️ Alias {alias} points to {targets}. Calls go there, not to {deployment}.")

### 5. Create a runtime API key

Apps authenticate to the gateway with a runtime API key, not with Foundry keys. The cell creates `app-first-call` if
it's missing, then reads its primary value with `listSecrets`. Notebooks 04 and 06 reuse the same key. The value
stays in memory: the notebook never prints it in full or writes it to a file.

In [ ]:
def get_or_create_key(name):
    """Create the runtime API key if it's missing, then read its primary value."""
    key_id = f"{GATEWAY_ID}/apiKeys/{name}"
    created = arm("GET", key_id) is None
    if created:
        arm("PUT", key_id, {"properties": {}})
    secrets = arm("POST", f"{key_id}/listSecrets")
    return secrets["primaryKey"], created

api_key, created = get_or_create_key(api_key_name)
print(("✅ Created" if created else "♻️ Reusing") + f" key {api_key_name}: {api_key[:4]}…")

### 6. Call the model with the OpenAI SDK

Point the OpenAI client at `<gateway-url>/default/models/openai/v1` and pass the runtime key as `api_key`. The SDK
sends it as `Authorization: Bearer <key>`, which the gateway accepts. Don't send both `api-key` and `Authorization`:
the gateway returns `400`. The `model` field holds the alias.

A brand-new key, model, or role assignment can take a few minutes to apply, so the helper retries `401`, `403`, `404`,
`429`, and `5xx` responses for up to 10 minutes.

In [ ]:
client = OpenAI(base_url=f"{gateway_url}/default/models/openai/v1", api_key=api_key)

def call_with_retry(call, minutes=10):
    deadline = time.time() + minutes * 60
    while True:
        try:
            return call()
        except APIStatusError as error:
            retryable = error.status_code in (401, 403, 404, 429) or error.status_code >= 500
            if not retryable or time.time() > deadline:
                raise
            print(f"⏳ HTTP {error.status_code}, waiting for the configuration to apply...")
            time.sleep(30)

response = call_with_retry(lambda: client.responses.create(model=alias, input="Write a one-sentence status update."))
print(f"📨 Responses API:        {response.output_text}")

chat = call_with_retry(lambda: client.chat.completions.create(
    model=alias, messages=[{"role": "user", "content": "Describe an AI gateway in one sentence."}]))
print(f"📨 Chat Completions API: {chat.choices[0].message.content}")
print(f"🔢 Tokens: {chat.usage.prompt_tokens} in, {chat.usage.completion_tokens} out")
assert response.output_text and chat.choices[0].message.content

### 7. Verify

Run the same checks as the lab guide:

- The model appears in the workspace's model list.
- A request with the key in the `api-key` header returns `200`.
- The same request without a key returns `401`. The gateway rejects it before any policy runs.
- A request for a name that isn't an alias returns `404` with `model_not_found`. Backend model and deployment names
  aren't callable unless an alias with that name exists.

In [ ]:
models = arm("GET", f"{WS}/models")["value"]
for m in models:
    print(f"🤖 {m['name']:<20} {m['properties'].get('providerKind')}")
assert any(m["name"] == deployment for m in models)

chat_url = f"{gateway_url}/default/models/openai/v1/chat/completions"
payload  = {"model": alias, "messages": [{"role": "user", "content": "ping"}]}

with_key = requests.post(chat_url, json=payload, headers={"api-key": api_key}, timeout=60)
print(f"📨 With key:      HTTP {with_key.status_code}")
assert with_key.status_code == 200, with_key.text[:500]

without_key = requests.post(chat_url, json=payload, timeout=60)
print(f"📨 Without key:   HTTP {without_key.status_code}")
assert without_key.status_code == 401

wrong_alias = requests.post(chat_url, json={**payload, "model": alias[:-1] + "x"}, headers={"api-key": api_key}, timeout=60)
print(f"📨 Wrong alias:   HTTP {wrong_alias.status_code} {wrong_alias.text[:120]}")
assert wrong_alias.status_code == 404 and "model_not_found" in wrong_alias.text
print("✅ All checks passed")

### 8. Clean up resources

Keep the gateway, model, alias, and key: notebooks 04, 06, and 11 reuse them. When you finish the series, uncomment
the lines below. Deleting the Foundry account soft-deletes it; the purge frees the name and stops any charges.

In [ ]:
# Notebooks 04, 06, and 11 reuse these. Uncomment to remove them when you're done with the series.
# arm("DELETE", f"{GATEWAY_ID}/apiKeys/{api_key_name}")
# arm("DELETE", f"{WS}/aliases/{alias}")
# arm("DELETE", MODEL_PATH)
# arm("DELETE", PROVIDER_PATH)
# arm("DELETE", role_assignment, api_version="2022-04-01")
# arm("DELETE", FOUNDRY_ID, api_version=COGNITIVE_API)
# DELETED_ID = (f"/subscriptions/{subscription_id}/providers/Microsoft.CognitiveServices/locations/{location}"
#               f"/resourceGroups/{foundry_rg}/deletedAccounts/{foundry_account}")
# while not arm("GET", DELETED_ID, api_version=COGNITIVE_API):   # wait for the soft-deleted account to appear
#     time.sleep(10)
# arm("DELETE", DELETED_ID, api_version=COGNITIVE_API)   # purge
# print("🗑️ Removed the key, alias, model, provider, role assignment, and Foundry account")
if arm("GET", FOUNDRY_ID, api_version=COGNITIVE_API):
    print(f"ℹ️ Kept {alias}, {provider}, key {api_key_name}, and {foundry_account} for the next notebooks")

## Troubleshooting

- **`401` on every call**: the key is new or wrong. Rerun step 5; new keys can take a minute to work. Send the key in
  `api-key` *or* `Authorization: Bearer`, not both (`400`).
- **`403` or `500` from the backend for several minutes**: the Foundry User assignment hasn't applied yet. Check it with
  `az role assignment list --scope <foundry-account-id> -o table`, then wait.
- **`404` with `model_not_found`**: `model` must match an alias exactly. List aliases with
  `arm("GET", f"{WS}/aliases")`.
- **`InsufficientQuota` when you create the deployment**: pick another region or lower `capacity`, or reuse an
  existing account by setting `foundry_account` and `foundry_rg`.
- **`CustomDomainInUse` or `FlagMustBeSetForRestore`**: a deleted account with the same name is in the soft-delete
  state. Purge it, or set another `foundry_account` name.
- **`429`**: the deployment's tokens-per-minute quota is used up. Wait a minute, or raise `capacity`.

## Next steps

- Continue with [Federate MCP tools behind one endpoint](../04-federate-mcp-tools/README.md) and its notebook [04-federate-mcp-tools.ipynb](../04-federate-mcp-tools/04-federate-mcp-tools.ipynb).
- Throttle callers with [token limits](../06-token-limits/README.md) and its notebook [06-token-limits.ipynb](../06-token-limits/06-token-limits.ipynb).
- Add other providers behind stable names in [lab 03](../03-multi-provider-aliases/README.md).
- Docs: [Access models through AI Gateway](https://ai.gateway.azure.com/docs/model-access).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/02-first-model-call/](https://azure.github.io/ai-gateway/labs/02-first-model-call/).